# RAG Database Generation for Colab

This notebook is a Colab-friendly version of `src/rag_pipeline.py` for building the Chroma vector database.

It intentionally does **not** use Ollama. The notebook only performs document loading, chunking, embedding, and Chroma persistence.

In [ ]:
%pip install -q chromadb llama-index-core llama-index-vector-stores-chroma llama-index-embeddings-huggingface sentence-transformers

## Workspace setup

Choose whether to work inside Google Drive or only inside the Colab runtime.

- Put your `.txt` and `.json` files under `documents/`.
- The generated Chroma database will be written under `data/chroma/`.
- If you use Drive, both the documents and the generated database will persist after the runtime stops.

In [1]:
from pathlib import Path

USE_GOOGLE_DRIVE = False
WORKSPACE_NAME = "C:/Users/felip/OneDrive/Documentos/cat_gpt"

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    workspace_root = Path("/content/drive/MyDrive") / WORKSPACE_NAME
else:
    workspace_root = Path("/content") / WORKSPACE_NAME

documents_dir = workspace_root / "documents"
data_root = workspace_root / "data"

documents_dir.mkdir(parents=True, exist_ok=True)
data_root.mkdir(parents=True, exist_ok=True)

print(f"Workspace: {workspace_root}")
print(f"Documents directory: {documents_dir}")
print(f"Output directory: {data_root}")

Workspace: C:\Users\felip\OneDrive\Documentos\cat_gpt
Documents directory: C:\Users\felip\OneDrive\Documentos\cat_gpt\documents
Output directory: C:\Users\felip\OneDrive\Documentos\cat_gpt\data


In [2]:
# from __future__ import annotations

from dataclasses import dataclass
import json
import re

from chromadb import PersistentClient
from llama_index.core import Document, Settings, StorageContext, VectorStoreIndex
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore


@dataclass(frozen=True)
class NotebookConfig:
    documents_dir: Path
    data_root: Path
    collection_name: str = "rag_documents"
    embedding_model: str = "BAAI/bge-m3"
    chunk_size: int = 1024
    chunk_overlap: int = 128

    def chroma_dir(self) -> Path:
        raw_name = f"{self.embedding_model}-{self.chunk_size}-{self.chunk_overlap}"
        safe_name = re.sub(r"[^A-Za-z0-9._-]+", "-", raw_name).strip("-")
        return self.data_root / "chroma" / safe_name


config = NotebookConfig(documents_dir=documents_dir, data_root=data_root)
embed_model = HuggingFaceEmbedding(model_name=config.embedding_model)
Settings.embed_model = embed_model

SUPPORTED_EXTENSIONS = {".txt", ".json"}


def discover_source_files(root: Path) -> list[Path]:
    if not root.exists():
        return []

    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )


def normalize_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def relative_path(path: Path, root: Path) -> str:
    try:
        return path.relative_to(root).as_posix()
    except ValueError:
        return path.as_posix()


def build_metadata(path: Path, root: Path, source_type: str, page_number: int | None = None) -> dict[str, object]:
    metadata: dict[str, object] = {
        "document_name": path.name,
        "relative_path": relative_path(path, root),
        "source_type": source_type,
    }
    if page_number is not None:
        metadata["page"] = page_number
    return metadata


def load_text(path: Path, root: Path) -> Document | None:
    text = normalize_text(path.read_text(encoding="utf-8", errors="replace"))
    if not text:
        return None
    return Document(text=text, metadata=build_metadata(path, root, "txt"))


def coerce_json_records(payload: object) -> list[dict[str, object]]:
    if isinstance(payload, list):
        return [record for record in payload if isinstance(record, dict)]

    if isinstance(payload, dict):
        for key in ("items", "documents", "chunks", "records"):
            value = payload.get(key)
            if isinstance(value, list):
                return [record for record in value if isinstance(record, dict)]
        return [payload]

    return []


def extract_json_text(record: dict[str, object]) -> str:
    preferred_keys = ("texto_embedding", "texto", "content", "body", "description")
    for key in preferred_keys:
        value = record.get(key)
        if isinstance(value, str) and value.strip():
            return normalize_text(value)

    parts: list[str] = []
    for key in ("titulo", "secao", "fonte", "url"):
        value = record.get(key)
        if isinstance(value, str) and value.strip():
            parts.append(f"{key}: {value.strip()}")

    for key, value in record.items():
        if key in {"id", "titulo", "secao", "fonte", "url", "tipo", "texto", "texto_embedding"}:
            continue
        if isinstance(value, str) and value.strip():
            parts.append(f"{key}: {value.strip()}")

    return normalize_text("\n".join(parts))


def load_json(path: Path, root: Path) -> list[Document]:
    try:
        payload = json.loads(path.read_text(encoding="utf-8", errors="replace"))
    except json.JSONDecodeError as exc:
        print(f"Skipped invalid JSON file: {path} ({exc})")
        return []

    documents: list[Document] = []
    for index, record in enumerate(coerce_json_records(payload)):
        text = extract_json_text(record)
        if not text:
            continue

        metadata = build_metadata(path, root, "json")
        metadata["json_record_index"] = index

        for key in ("id", "titulo", "secao", "fonte", "url", "tipo"):
            metadata[key] = str(record.get(key, "")).strip()

        document = Document(text=text, metadata=metadata)
        documents.append(document)
        print(f"Loaded JSON record: {path} [{index}] ({len(document.text)} chars)")

    return documents


def load_documents(root: Path) -> list[Document]:
    documents: list[Document] = []

    for path in discover_source_files(root):
        if path.suffix.lower() == ".json":
            documents.extend(load_json(path, root))
            continue

        document = load_text(path, root)
        if document is not None:
            documents.append(document)
            print(f"Loaded document: {path} ({len(document.text)} chars)")

    return documents


def chunk_documents(documents: list[Document], chunk_size: int, chunk_overlap: int):
    splitter = TokenTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    nodes = splitter.get_nodes_from_documents(documents)

    for index, node in enumerate(nodes):
        node.metadata["chunk_index"] = index
        node.metadata["chunk_id"] = node.metadata.get(
            "chunk_id",
            f"{node.metadata.get('relative_path', 'document')}::chunk-{index:05d}",
        )

    print(f"Split {len(documents)} documents into {len(nodes)} chunks")
    return nodes


def clear_collection(client: PersistentClient, persist_dir: Path, collection_name: str) -> None:
    existing_names = {collection.name for collection in client.list_collections()}
    if collection_name in existing_names:
        client.delete_collection(collection_name)
        print(f"Cleared existing collection '{collection_name}' in {persist_dir}")


def build_vector_store(client: PersistentClient, persist_dir: Path, collection_name: str):
    collection = client.get_or_create_collection(name=collection_name, metadata={"hnsw:space": "cosine"})
    vector_store = ChromaVectorStore(chroma_collection=collection)
    print(f"Created or loaded collection '{collection_name}' in {persist_dir}")
    return vector_store, collection


def build_index() -> Path:
    print(f"Loading documents from {config.documents_dir}")
    documents = load_documents(config.documents_dir)

    if not documents:
        raise ValueError(
            f"No supported documents found in {config.documents_dir}. Add .txt or .json files before running again."
        )

    nodes = chunk_documents(documents, config.chunk_size, config.chunk_overlap)
    chroma_dir = config.chroma_dir()
    chroma_dir.mkdir(parents=True, exist_ok=True)

    # Keep a single PersistentClient alive for the whole build (clear + create + insert).
    # Opening a fresh client per helper call lets Chroma's shared-system refcounting
    # tear down the underlying system while later inserts are still in flight,
    # corrupting the persisted vector index.
    client = PersistentClient(path=str(chroma_dir))

    clear_collection(client, chroma_dir, config.collection_name)
    vector_store, collection = build_vector_store(client, chroma_dir, config.collection_name)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    print(f"Seeding vector store in {chroma_dir} with collection '{config.collection_name}'")
    print(f"Indexing {len(nodes)} chunks into vector store...")
    VectorStoreIndex(nodes=nodes, storage_context=storage_context, embed_model=embed_model, show_progress=True)

    print(
        f"Indexed {len(documents)} source documents into {len(nodes)} chunks and persisted {collection.count()} vectors in {chroma_dir}"
    )
    return chroma_dir

c:\Users\felip\AppData\Local\pypoetry\Cache\virtualenvs\cat-gpt-x-qARWO5-py3.11\Lib\site-packages\pydantic\_internal\_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\felip\AppData\Local\pypoetry\Cache\virtualenvs\cat-gpt-x-qARWO5-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3055

In [3]:
chroma_dir = build_index()
print(f"Database ready at: {chroma_dir}")

Loading documents from C:\Users\felip\OneDrive\Documentos\cat_gpt\documents
Loaded JSON record: C:\Users\felip\OneDrive\Documentos\cat_gpt\documents\aaha_feline_life_stage_chunks.json [0] (4598 chars)
Loaded JSON record: C:\Users\felip\OneDrive\Documentos\cat_gpt\documents\aaha_feline_life_stage_chunks.json [1] (4882 chars)
Loaded JSON record: C:\Users\felip\OneDrive\Documentos\cat_gpt\documents\aaha_feline_life_stage_chunks.json [2] (1782 chars)
Loaded JSON record: C:\Users\felip\OneDrive\Documentos\cat_gpt\documents\aaha_feline_life_stage_chunks.json [3] (3177 chars)
Loaded JSON record: C:\Users\felip\OneDrive\Documentos\cat_gpt\documents\aaha_feline_life_stage_chunks.json [4] (2215 chars)
Loaded JSON record: C:\Users\felip\OneDrive\Documentos\cat_gpt\documents\aaha_feline_life_stage_chunks.json [5] (2851 chars)
Loaded JSON record: C:\Users\felip\OneDrive\Documentos\cat_gpt\documents\aaha_feline_life_stage_chunks.json [6] (2326 chars)
Loaded JSON record: C:\Users\felip\OneDrive\Docum

Generating embeddings: 100%|██████████| 1254/1254 [1:33:11<00:00,  4.46s/it]  


Indexed 1248 source documents into 1254 chunks and persisted 1254 vectors in C:\Users\felip\OneDrive\Documentos\cat_gpt\data\chroma\BAAI-bge-m3-1024-128
Database ready at: C:\Users\felip\OneDrive\Documentos\cat_gpt\data\chroma\BAAI-bge-m3-1024-128


## Optional: download the generated database

Run the next cell only if you want a zip file of the generated Chroma directory.

In [ ]:
import shutil

archive_path = shutil.make_archive(str(chroma_dir), "zip", root_dir=chroma_dir)
print(f"Created archive: {archive_path}")

try:
    from google.colab import files

    files.download(archive_path)
except ImportError:
    print("google.colab is not available in this environment. Download the zip file manually if needed.")